# EUR/USD anomaly detection with path signatures: new FX channels

A cleaned, FX-only version of `anomaly/anomaly_experiments.ipynb`. Same question: does a
signature-based anomaly score tell us something about the coming period **beyond** what a volatility
model already knows? An anomaly is judged against *surprise*: realized volatility far above the HAR
forecast (the top 20% of log(realized / forecast)), and scored within strata of the HAR forecast so a
detector gets no credit for just finding high-volatility periods.

What changed from the original notebook:

- **Channels are the one thing that varies** (5 sets), including the new FX channels: time-of-day-adjusted
  price and range, and the abnormal bid-ask spread (see `vol_fx.ipynb`, section 1).
- **Everything else is fixed in advance**: windows normalized by the volatility before them (so detectors
  see shape, not level), one window length (one trading week for 1 day, one trading month for 5 days),
  lead-lag paths, signature depths 2 and 3. The original grid had about 250 configs, which overfit the
  selection.
- **Detectors**: Mahalanobis and Isolation Forest on every config, then the HMM on the top 3 signature
  configs and the best config of each control. The autoencoder is dropped to save time (it is the slowest detector).
- No crypto branch, no add-the-HMM-afterwards cell.

Run this before section 7 of `vol_fx.ipynb`, which uses the detectors selected here.

In [ ]:
import logging

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import fx_lab as L
from anomaly import report as R, run as RUN

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S", force=True)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

QUICK = False  # True: data from 2021 and fewer bootstrap draws, to check that everything runs

## 1. Configuration

| Channel set | Channels |
|---|---|
| `price` | log price (the original reference) |
| `price+volume+range+flow` | the original full set (FX "flow" is the tick rule, so weak) |
| `price_sa` | time-of-day-adjusted log price |
| `price_sa+spread` | plus the abnormal spread |
| `price_sa+volume+range_sa+spread` | everything new |

Each channel set also gives the signature-free controls built from the same window: a lag bank with as
many columns as the signature, summary stats, and raw block sums. Two volatility baselines (the current
level and the HAR forecast) are scored too.

In [ ]:
cfg = L.anomaly_config(quick=QUICK)
print(len(cfg.rep_configs()), "path recipes per horizon; data", cfg.start, "to", cfg.end)
cfg

## 2. Run

Each horizon runs in its own process. Finished path recipes are saved under `parts/`, so an interrupted
run resumes where it stopped; finished horizons (`meta.json`) are skipped. Delete
`results/anomaly/<horizon>/` to redo one.

In [ ]:
root = RUN.run_all(cfg, subprocess_per_horizon=True)
res = R.load_results(root)
pd.DataFrame(res.meta).T

## 3. The gate: does the best signature config beat everything signature-free?

The best signature config and the best signature-free config are chosen on folds 1-2 (`select`) and
checked on the last fold (`confirm`). The gate passes if the signature config is ahead on both, with a
block-bootstrap one-sided p < 0.05 on the last fold. 0.5 = no information beyond the HAR forecast.

In [ ]:
display(R.gate_table(res).style.format(precision=3))

## 4. Which channels help?

The best signature config of each channel set, per detector. Compare `price` with `price_sa` (the effect
of the time-of-day adjustment) and `price_sa` with `price_sa+spread` (the spread). `Select` is on the
selection folds, `Confirm` on the last fold; differences of a few hundredths are within noise.

In [ ]:
display(L.channel_table(res).style.format(precision=3))
display(R.leaderboard(res, top=10).style.format(precision=3))

## 5. Synthetic check: shape-only anomalies

Test windows whose returns are re-ordered into a V or an inverted V: volatility, net move, skew and
kurtosis unchanged, only the shape differs. Shows what each selected detector *can* see in principle;
it is not evidence about real markets.

In [ ]:
display(R.synthetic_table(res).style.format(precision=3))

## 6. Scores over time

Score percentiles of the selected detectors, with labeled FX events shaded (a case study only).

In [ ]:
for hk in res.meta:
    R.plot_timeline(res, hk); plt.show()

## 7. Compared with the previous FX anomaly run

The previous run (`anomaly/results/fx/anomaly`: the original channels, the full grid) next to this one.
Different grids and end dates, so this compares the conclusions, not the numbers row by row.

In [ ]:
prev_root = L.ROOT / "anomaly" / "results" / "fx" / "anomaly"
try:
    prev = R.load_results(prev_root)
    both = pd.concat({"previous": R.gate_table(prev), "new": R.gate_table(res)}, names=["Run"]).reset_index(0)
    display(both.style.format(precision=3))
except FileNotFoundError:
    print("No previous FX anomaly run found at", prev_root)

## 8. Next

`best_configs.json` in this run's folder now holds the selected detectors. Run section 7 of
`vol_fx.ipynb` to use them in the volatility forecasts.